In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/targe-prismatic/{runs,data,hf_cache}

# Confirm it reports an L4/A100 (not T4) if doing 7B. T4 is OK for Phi-2 smoke.

/bin/bash: line 1: nvidia-smi: command not found
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
"""
Install dependencies
"""
!pip install -e . --quiet
# !pip install flash-attn --no-build-isolation --quiet  # optional but speeds things up

[Errno 2] No such file or directory: 'drive/MyDrive/targe-prismatic-vlms/'
/content/drive/MyDrive/targe-prismatic-vlm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for prismatic (pyproject.toml) ... done


In [15]:
"""
Count folders in directory
"""
import os

# 1. Update this to the exact subfolder path inside your Drive
folder_path = '/content/drive/MyDrive/targe-prismatic-vlms/data/download/llava-laion-cc-sbu-558k'

# 2. Efficiently count files
file_count = 0
for root, dirs, files in os.walk(folder_path):
    file_count += len(files)

print(f"Total number of files: {file_count:,}")

Total number of files: 88,876


In [12]:
import os
from google.colab import userdata
from huggingface_hub import login

os.environ["HF_HOME"] = "/content/drive/MyDrive/targe-prismatic-vlms/hf_cache"
hf_token = userdata.get('hf_token')

!echo hf_token > /content/drive/MyDrive/targe-prismatic-vlm/.hf_token
login(token=hf_token)

/bin/bash: line 1: /content/drive/MyDrive/targe-prismatic-vlm/.hf_token: No such file or directory


In [ ]:
# !python scripts/make_smoke_dataset.py --num_samples 200000 --train_pct 0.9 --root_dir data --seed 7

In [ ]:
  # import json
  # from pathlib import Path
  # from concurrent.futures import ThreadPoolExecutor

  # # --- edit these three ---
  # SRC       = Path("data/download/llava-laion-cc-sbu-558k/chat.json")
  # DST       = Path("data/download/llava-laion-cc-sbu-558k/chat.pruned.json")
  # IMAGE_DIR = Path("data/download/llava-laion-cc-sbu-558k")
  # # ------------------------

  # text = SRC.read_text()
  # is_jsonl = SRC.suffix == ".jsonl"
  # entries = (
  #     [json.loads(l) for l in text.splitlines() if l.strip()]
  #     if is_jsonl else json.loads(text)
  # )

  # def keep(e):
  #     img = e.get("image")
  #     return True if not img else (IMAGE_DIR / img).is_file()

  # with ThreadPoolExecutor(max_workers=32) as pool:
  #     mask = list(pool.map(keep, entries))

  # kept = [e for e, ok in zip(entries, mask) if ok]

  # DST.parent.mkdir(parents=True, exist_ok=True)
  # if is_jsonl:
  #     with DST.open("w") as f:
  #         for e in kept:
  #             f.write(json.dumps(e) + "\n")
  # else:
  #     DST.write_text(json.dumps(kept))

  # print(f"kept {len(kept)}/{len(entries)}  ({len(entries) - len(kept)} dropped)")

kept 88874/558128  (469254 dropped)


In [ ]:
# 1. Copy the zipped dataset from Drive to the local instance
!rsync -ah --info=progress2 /content/drive/MyDrive/targe-prismatic-vlm/data/download/llava-laion-cc-sbu-558k/images.zip /content/

# 2. Unzip it into the local /content/data directory
!unzip -q /content/images.zip -d /content/data/

# 3. (Optional) Keep your 'runs' folder symlinked so model checkpoints save directly to Drive
!rm -rf runs && ln -s /content/drive/MyDrive/targe-prismatic-vlms/runs runs

^C
[/content/images.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/images.zip or
        /content/images.zip.zip, and cannot find /content/images.zip.ZIP, period.
ln: failed to create symbolic link 'runs': Operation not supported


In [ ]:
!rm -rf data && ln -s /content/drive/MyDrive/targe-prismatic/data data
!rm -rf runs && ln -s /content/drive/MyDrive/targe-prismatic/runs runs

In [ ]:
torchrun --standalone --nnodes 1 --nproc-per-node 1 scripts/pretrain.py \
  --model.type "targe-smollm2-360m-clipb-224px" \
  --stage align \
  --model.align_per_device_batch_size 14 \
  --model.align_learning_rate 1e-4 \
  --model.align_global_batch_size 14 \
  --model.align_max_steps 500 \
  --dataset.type "llava-v15" \
  --run_id "targe-smollm2-next" \
  --trackers '[jsonl,wandb]' \
  --wandb_entity nbusich-northeastern-university \
  --wandb_project targe


05/18 [05:57:22] INFO     | >> [*] Prismatic VLM Training ::     ]8;id=882004;file:///content/drive/MyDrive/targe-prismatic-vlm/scripts/pretrain.py\pretrain.py]8;;\:]8;id=711187;file:///content/drive/MyDrive/targe-prismatic-vlm/scripts/pretrain.py#126\126]8;;\
                          Gathering Light                                       
                 INFO     | >>     |=> "Life is like a prism;    ]8;id=573135;file:///content/drive/MyDrive/targe-prismatic-vlm/scripts/pretrain.py\pretrain.py]8;;\:]8;id=804250;file:///content/drive/MyDrive/targe-prismatic-vlm/scripts/pretrain.py#140\140]8;;\
                          what you see depends on how you turn                  
                          the glass."                                           
                 INFO     | >> [*] Loading Vision Backbone       ]8;id=50631;file:///content/drive/MyDrive/targe-prismatic-vlm/scripts/pretrain.py\pretrain.py]8;;\:]8;id=75954;file:///content/drive/MyDrive/targe-pr